# Multi-turn policy comparison (calming vs. provocative)

Runs short 5-turn conversations across all seven emotions with three deterministic policies (calming, provocative, always-validate baseline). Saves per-turn logs, summaries, and heatmaps under `results/multiturn_runs/`. Set `OPENAI_API_KEY` before running; GPU is optional but recommended.

In [ ]:
import os
from pathlib import Path
import pandas as pd
from dynamic_conversation import (
    EmotionFlowAnalyzer,
    MultiTurnRollout,
    calming_policy,
    provocative_policy,
    always_validate_policy,
)

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# Configuration
emotions = EmotionFlowAnalyzer.EMOTIONS
policies = {
    "calming": calming_policy,
    "provocative": provocative_policy,
    "validate": always_validate_policy,
}
runs_per_emotion = 3  # keep small to finish within ~20–30 minutes
turns = 5
style_modifier = "concise and emotionally attuned"
out_dir = Path("results/multiturn_runs")
out_dir.mkdir(parents=True, exist_ok=True)

sim = MultiTurnRollout(
    use_gpu=True,
    turns=turns,
    default_style=style_modifier,
)
print(f"Device set to GPU? {sim.emotion_analyzer.classifier.device != -1}")

In [ ]:
# Run all policies
per_policy = {}
for name, fn in policies.items():
    csv_path = out_dir / f"{name}_5turn.csv"
    plot_path = out_dir / f"{name}_5turn_heatmap.png"
    per_turn_df, summary_df = sim.run_policy_batch(
        policy_name=name,
        policy_fn=fn,
        emotions=emotions,
        runs_per_emotion=runs_per_emotion,
        turns=turns,
        style_modifier=style_modifier,
        use_llm_seed=True,
        use_esconv_seed=False,
        save_csv=csv_path,
        save_plot=plot_path,
    )
    per_policy[name] = (per_turn_df, summary_df)
    display(summary_df.head())
print("✓ Completed all policy runs")

In [ ]:
# Simple comparisons: final emotion distribution and trajectory means
rows = []
for name, (turn_df, summary_df) in per_policy.items():
    if turn_df.empty:
        continue
    last_turn = turn_df['turn'].max()
    finals = turn_df[turn_df['turn'] == last_turn]
    dist = finals['detected_emotion'].value_counts(normalize=True)
    traj_mean = summary_df['trajectory_to_intended'].mean() if not summary_df.empty else 0.0
    rows.append({
        'policy': name,
        'trajectory_to_intended_mean': traj_mean,
        'final_top_emotion': dist.idxmax() if not dist.empty else 'n/a',
        'final_top_prop': dist.max() if not dist.empty else 0.0,
    })
comparison_df = pd.DataFrame(rows).sort_values(by='trajectory_to_intended_mean', ascending=False)
display(comparison_df)

## Notes
- Outputs: per-turn logs at `results/multiturn_runs/<policy>_5turn.csv`, summaries at `.summary.csv`, heatmaps at `_5turn_heatmap.png`.
- `trajectory_to_intended` scores weight later turns; higher means a stronger pull toward the starting emotion.
- Adjust `runs_per_emotion` or `turns` if runtime is too high; keep seeds consistent across policies by controlling randomness in future iterations.

## Trajectory CIs by start emotion × policy

Computes conversation-level trajectory scores for every target emotion, grouped by (start emotion, policy), and reports mean ± 95% CI. Uses per-turn emotion scores already saved under `results/multiturn_runs/`.


In [ ]:
import json
import numpy as np

# Load per-policy per-turn and summary data (from current run or disk)
if 'per_policy' in globals() and per_policy:
    turn_frames = [v[0] for v in per_policy.values() if not v[0].empty]
    summary_frames = [v[1] for v in per_policy.values() if not v[1].empty]
else:
    turn_frames = []
    summary_frames = []
    for csv_file in out_dir.glob('*_5turn.csv'):
        turn_frames.append(pd.read_csv(csv_file))
        summary_file = csv_file.with_suffix('.summary.csv')
        if summary_file.exists():
            summary_frames.append(pd.read_csv(summary_file))

if not turn_frames or not summary_frames:
    raise RuntimeError('No per-turn or summary data found; run the policy batch first.')

turns_df = pd.concat(turn_frames, ignore_index=True)
summ_df = pd.concat(summary_frames, ignore_index=True)
lookup = summ_df.set_index('conversation_id')

emotions = EmotionFlowAnalyzer.EMOTIONS

def compute_weighted_trajectories(group):
    group = group.sort_values('turn')
    n = len(group)
    weight_sum = sum((i+1)/n for i in range(n))
    agg = {e: 0.0 for e in emotions}
    for i, row in enumerate(group.itertuples(index=False)):
        w = (i + 1) / n
        scores = json.loads(row.emotion_scores) if isinstance(row.emotion_scores, str) else {}
        for e in emotions:
            agg[e] += w * scores.get(e, 0.0)
    return {e: (agg[e] / weight_sum if weight_sum else 0.0) for e in emotions}

traj_rows = []
for cid, group in turns_df.groupby('conversation_id'):
    if cid not in lookup.index:
        continue
    meta = lookup.loc[cid]
    traj = compute_weighted_trajectories(group)
    traj_rows.append({
        'conversation_id': cid,
        'policy': meta['policy'],
        'start_emotion': meta['intended_emotion'],
        **traj,
    })

traj_df = pd.DataFrame(traj_rows)

# Aggregate mean ± 95% CI per (start emotion, policy, target emotion)
rows = []
for (start_emotion, policy), subset in traj_df.groupby(['start_emotion', 'policy']):
    for target in emotions:
        vals = subset[target].dropna()
        n = len(vals)
        mean = vals.mean() if n else 0.0
        if n > 1:
            se = vals.std(ddof=1) / np.sqrt(n)
            ci = 1.96 * se
        else:
            ci = 0.0
        rows.append({
            'start_emotion': start_emotion,
            'policy': policy,
            'target_emotion': target,
            'mean': mean,
            'ci': ci,
        })

agg = pd.DataFrame(rows)
agg['mean_ci'] = agg.apply(lambda r: f"{r['mean']:.3f} ± {r['ci']:.3f}", axis=1)

pivot = agg.pivot_table(index=['start_emotion','policy'], columns='target_emotion', values='mean_ci', aggfunc='first')
# also keep numeric means for highlighting
mean_pivot = agg.pivot_table(index=['start_emotion','policy'], columns='target_emotion', values='mean', aggfunc='first')

styled = pivot.style.format(None).apply(lambda row: ['font-weight: bold' if val==row.max() else '' for val in mean_pivot.loc[row.name]], axis=1)
display(styled)

